# Xây dựng mô hình dự đoán kết quả bệnh nhân mắc bệnh tiểu đường trên dataset Pima Indians Diabetes

## 1. Import các thư viện cần thiết

In [26]:
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
from scipy.stats import randint, uniform

# model selection
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

# algorithms
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
import sklearn

from xgboost import XGBClassifier

# metrics
from sklearn.metrics import accuracy_score , ConfusionMatrixDisplay, confusion_matrix, f1_score, roc_auc_score, classification_report

import warnings

## 2. Thiết lập tham số

In [27]:
parameters = {}
parameters["random_state"] = 42
parameters["k_fold"] = 10
parameters["n_random_search"] = 50

## 3. Lấy dataset đã được dọn dẹp và chuẩn hóa.

In [28]:
X_train = pd.DataFrame(np.load("../exps/data/X_train_scaled.npy"))
y_train = pd.DataFrame(np.load("../exps/data/y_train.npy"))
X_train = X_train.squeeze()
y_train = y_train.squeeze()

X_test = pd.DataFrame(np.load("../exps/data/X_test_scaled.npy"))
y_test = pd.DataFrame(np.load("../exps/data/y_test.npy"))
X_test = X_test.squeeze()
y_test = y_test.squeeze()

## 4. Kiểm tra dữ liệu

In [29]:
X_train.head()

,0,1,2,3,4,5
0,0.091816,1.509134,0.232332,-0.855792,0.060909,-0.937699
1,0.819760,0.166432,0.275548,0.219850,0.060909,0.923014
2,-0.282398,1.580089,1.634622,-0.855792,1.192643,0.662844
3,1.721401,1.028798,-0.810374,0.951417,1.319553,0.917181
4,-1.711465,2.107460,1.272717,-0.381319,-0.428882,0.200399


In [30]:
y_train.head()

0    1
1    0
2    1
3    1
4    1
Name: 0, dtype: int64

In [31]:
X_test.head()

,0,1,2,3,4,5
0,-1.711465,1.917549,0.374947,0.802700,2.136763,-0.030104
1,0.382080,0.166432,0.040584,0.309908,-0.780422,-0.825212
2,-0.809824,0.422981,-1.397126,-1.264453,-0.601948,-0.194678
3,0.619242,0.199392,0.332591,0.565790,0.215262,-0.849994
4,0.091816,0.974805,-1.971610,0.482751,0.365553,-0.904107


In [32]:
y_test.head()

0    1
1    1
2    0
3    1
4    0
Name: 0, dtype: int64

## 5. Chia K-Fold

In [33]:
skf = StratifiedKFold(n_splits=parameters["k_fold"], shuffle=True, random_state=parameters["random_state"])

In [34]:
param_distributions = {}
param_distributions["Logistic Regression"] =  {
        "C": uniform(0.01, 10),        
        "penalty": ["l2", None],
        "solver": ["lbfgs", "saga"],
        "max_iter": randint(1000, 2000),
    }

param_distributions["Random Forest"] = {
        "n_estimators": randint(100, 200),
        "max_depth": [None],
        "min_samples_split": randint(2, 11),
        "min_samples_leaf": randint(1, 5),
        "max_features": ["sqrt", "log2"],
        "bootstrap": [True, False],
    }

param_distributions["SVC"] = {
        "C": uniform(0.1, 10),
        "gamma": ["scale", 0.1, 0.01],
        "kernel": ["rbf", "sigmoid"],
        "probability": [True],
    }

param_distributions["K-Neighbours"] = {
        "n_neighbors": randint(3, 15),
        "weights": ["uniform", "distance"],
        "p": [1, 2],
    }

param_distributions["Decision Tree"] = {
        "max_depth": [3, 5, 7, 9, None],
        "min_samples_split": randint(2, 11),
        "min_samples_leaf": randint(1, 6),
        "criterion": ["gini", "entropy", "log_loss"],
    }


## 6. Chọn các model để train.
Các model được chọn bao gồm: Logistic Regression, Random Forest, SVC, K-Neighbours, Decision Tree.

In [35]:
models = {}
models["Logistic Regression"] = LogisticRegression(random_state=parameters["random_state"])
                                
models["Random Forest"] = RandomForestClassifier(random_state=parameters["random_state"])

models["SVC"] = SVC(random_state=parameters["random_state"])

models["K-Neighbours"] = KNeighborsClassifier()

models["Decision Tree"] = DecisionTreeClassifier(random_state=parameters["random_state"])


## 7. Train các model trên tập dữ liệu train.

In [36]:
results = {name: {'accuracy': [], 'f1': [], 'roc_auc': []} for name in models.keys()}
best_models = {}


for name, model in models.items():
    model_random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions[name],
        n_iter=parameters["n_random_search"],
        scoring="accuracy",
        cv=parameters["k_fold"],
        n_jobs=-1,
        random_state=parameters["random_state"],
        verbose=1
    )

    model_random_search.fit(X_train, y_train)

    best_models[name] = model_random_search.best_estimator_

    

Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits


## 8. Tạo ra một voting model dùng Voting Ensemble

In [37]:
base_learners = [(name, best_models[name]) for name in best_models.keys()] # model tối ưu

voting_model = VotingClassifier(
    estimators=base_learners,
    voting='soft',
    n_jobs=-1
)

## 9. Kiểm tra hiệu suất voting model trên tập test.

In [38]:
voting_model.fit(X_train, y_train)
y_pred = voting_model.predict(X_test)    
y_prob = voting_model.predict_proba(X_test)[:,1]

print("\nPredicted values (first 10):")
print(y_pred[:10].tolist())

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confudsion Matrix:")
print(confusion_matrix(y_test, y_pred))



Predicted values (first 10):
[1, 0, 0, 0, 0, 0, 1, 0, 1, 0]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.87      0.84        76
           1       0.71      0.62      0.67        40

    accuracy                           0.78       116
   macro avg       0.76      0.75      0.75       116
weighted avg       0.78      0.78      0.78       116

Confudsion Matrix:
[[66 10]
 [15 25]]


## 10. Lưu file dưới dạng HTML.

In [39]:
import nbformat
from nbconvert import HTMLExporter

# load notebook
nb = nbformat.read("model4.ipynb", as_version=4)

# tạo exporter
html_exporter = HTMLExporter()
html_exporter.template_name = 'lab'

(body, resources) = html_exporter.from_notebook_node(nb)

# ghi file HTML
with open("../exps/model1/model_n.html", "w", encoding="utf-8") as f:
    f.write(body)

# Kết thúc